# Imports and helpers

In [2]:
from pathlib import Path
import json
import re
from typing import Any

import pandas as pd
from datasets import load_dataset
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

NOTEBOOK_DIR = Path.cwd()

# If notebook is in multilingual/notebooks, this gives repo root.
REPO_ROOT = NOTEBOOK_DIR.parents[1]
MULTILINGUAL_DIR = REPO_ROOT / "multilingual"

PROCESSED_ROOT = MULTILINGUAL_DIR / "data" / "processed" / "mera"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:    ", NOTEBOOK_DIR)
print("REPO_ROOT:       ", REPO_ROOT)
print("MULTILINGUAL_DIR:", MULTILINGUAL_DIR)
print("PROCESSED_ROOT:  ", PROCESSED_ROOT)

NOTEBOOK_DIR:     /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/notebooks
REPO_ROOT:        /scratch/workspace/emma/standard-project-m2-mystery-machine
MULTILINGUAL_DIR: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual
PROCESSED_ROOT:   /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera


In [3]:
HF_DATASET = "MERA-evaluation/MERA"

SELECTED_TASKS = [
    "use",
    "mamuramu",
    "mathlogicqa",
    "ruopenbookqa",
    "ruworldtree",
    "parus",
    "rcb",
    "rwsd",
]

# we do not use test/public_test for now.
ALLOWED_SPLITS = {"train", "validation", "dev"}

LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
CORRUPTION_RE = re.compile(r"[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f]")

In [4]:
def has_corruption(x: Any) -> bool:
    return bool(CORRUPTION_RE.search(str(x)))


def clean_text(x: Any) -> str:
    if x is None:
        return ""
    return str(x).strip()


def to_json_str(x: Any) -> str:
    return json.dumps(x, ensure_ascii=False)


def safe_get(d: Any, key: str, default: Any = "") -> Any:
    if isinstance(d, dict):
        return d.get(key, default)
    return default


def format_abcd(question: str, options: dict[str, Any]) -> str:
    return (
        f"{question}\n\n"
        f"A) {clean_text(options.get('A'))}\n"
        f"B) {clean_text(options.get('B'))}\n"
        f"C) {clean_text(options.get('C'))}\n"
        f"D) {clean_text(options.get('D'))}"
    )


def make_record(
    *,
    subset: str,
    split: str,
    source_id: str,
    task_type: str,
    prompt: str,
    answer: str,
    metadata: dict[str, Any],
) -> dict[str, Any]:
    return {
        "dataset": "mera",
        "subset": subset.lower(),
        "language": "russian",
        "split": split,
        "source_id": source_id,
        "task_type": task_type,
        "prompt": clean_text(prompt),
        "answer": clean_text(answer),
        "metadata": metadata,
    }


def is_valid_record(r: dict[str, Any]) -> bool:
    fields = [r["prompt"], r["answer"]]
    metadata_str = to_json_str(r.get("metadata", {}))

    return (
        r["split"] in {"train", "val"}
        and r["prompt"] != ""
        and r["answer"] != ""
        and not any(has_corruption(x) for x in fields)
        and not has_corruption(metadata_str)
    )

## Task-specific helpers for the subsets

In [5]:
def format_mamuramu(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    question = clean_text(inp.get("text"))
    options = {
        "A": inp.get("option_a"),
        "B": inp.get("option_b"),
        "C": inp.get("option_c"),
        "D": inp.get("option_d"),
    }

    return make_record(
        subset="MaMuRAMu",
        split=split,
        source_id=f"MaMuRAMu_{split}_{idx}",
        task_type="mcq",
        prompt=format_abcd(question, options),
        answer=clean_text(ex["outputs"]).upper(),
        metadata={
            "subject": clean_text(inp.get("subject")),
            "domain": clean_text(meta.get("domain")),
            "id": meta.get("id", idx),
        },
    )


def format_mathlogicqa(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    question = clean_text(inp.get("text"))
    options = {
        "A": inp.get("option_a"),
        "B": inp.get("option_b"),
        "C": inp.get("option_c"),
        "D": inp.get("option_d"),
    }

    return make_record(
        subset="MathLogicQA",
        split=split,
        source_id=f"MathLogicQA_{split}_{idx}",
        task_type="mcq",
        prompt=format_abcd(question, options),
        answer=clean_text(ex["outputs"]).upper(),
        metadata={
            "task": clean_text(meta.get("task")),
            "id": meta.get("id", idx),
        },
    )


def format_ruopenbookqa(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    question = clean_text(inp.get("text") or inp.get("question"))
    options = {
        "A": inp.get("option_a"),
        "B": inp.get("option_b"),
        "C": inp.get("option_c"),
        "D": inp.get("option_d"),
    }

    return make_record(
        subset="ruOpenBookQA",
        split=split,
        source_id=f"ruOpenBookQA_{split}_{idx}",
        task_type="mcq",
        prompt=format_abcd(question, options),
        answer=clean_text(ex["outputs"]).upper(),
        metadata={
            "id": meta.get("id", idx),
        },
    )


def format_ruworldtree(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    question = clean_text(inp.get("question") or inp.get("text"))
    options = {
        "A": inp.get("option_a"),
        "B": inp.get("option_b"),
        "C": inp.get("option_c"),
        "D": inp.get("option_d"),
    }

    return make_record(
        subset="ruWorldTree",
        split=split,
        source_id=f"ruWorldTree_{split}_{idx}",
        task_type="mcq",
        prompt=format_abcd(question, options),
        answer=clean_text(ex["outputs"]).upper(),
        metadata={
            "id": meta.get("id", idx),
            "exam_name": clean_text(meta.get("exam_name")),
            "school_grade": meta.get("school_grade", None),
            "knowledge_type": clean_text(meta.get("knowledge_type")),
        },
    )


def format_parus(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    q_type = clean_text(meta.get("task"))
    if q_type == "cause":
        q_line = "Что является причиной описанной ситуации?"
    elif q_type == "effect":
        q_line = "Что является следствием описанной ситуации?"
    else:
        q_line = "Какой вариант наиболее правдоподобен?"

    raw_answer = clean_text(ex["outputs"])
    answer_map = {"1": "A", "2": "B"}
    if raw_answer not in answer_map:
        raise ValueError(f"Unexpected PARus answer: {raw_answer!r}")

    prompt = (
        f"{clean_text(inp.get('premise'))}\n\n"
        f"{q_line}\n\n"
        f"A) {clean_text(inp.get('choice1'))}\n"
        f"B) {clean_text(inp.get('choice2'))}"
    )

    return make_record(
        subset="parus",
        split=split,
        source_id=f"parus_{split}_{idx}",
        task_type="mcq",
        prompt=prompt,
        answer=answer_map[raw_answer],
        metadata={
            "task": q_type,
            "id": meta.get("id", idx),
            "raw_answer": raw_answer,
            "option_a": clean_text(inp.get("choice1")),
            "option_b": clean_text(inp.get("choice2")),
        },
    )

def format_rcb(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    raw_answer = clean_text(ex["outputs"])
    answer_map = {"1": "A", "2": "B", "3": "C"}
    if raw_answer not in answer_map:
        raise ValueError(f"Unexpected RCB answer: {raw_answer!r}")

    prompt = (
        f"Ситуация:\n{clean_text(inp.get('premise'))}\n\n"
        f"Гипотеза:\n{clean_text(inp.get('hypothesis'))}\n\n"
        "Выберите отношение гипотезы к ситуации:\n"
        "A) гипотеза следует из ситуации\n"
        "B) гипотеза противоречит ситуации\n"
        "C) гипотеза независима от ситуации"
    )

    return make_record(
        subset="rcb",
        split=split,
        source_id=f"rcb_{split}_{idx}",
        task_type="mcq",
        prompt=prompt,
        answer=answer_map[raw_answer],
        metadata={
            "genre": clean_text(meta.get("genre")),
            "verb": clean_text(meta.get("verb")),
            "negation": clean_text(meta.get("negation")),
            "id": meta.get("id", idx),
            "raw_answer": raw_answer,
            "option_a": "гипотеза следует из ситуации",
            "option_b": "гипотеза противоречит ситуации",
            "option_c": "гипотеза независима от ситуации",
        },
    )


# NB: rwsd is binary choice (Yes/No answers), we reformat this to MCQ (2 options), Yes --> A since Y/N pretty balanced, roughly 50/50
def format_rwsd(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    raw_answer = clean_text(ex["outputs"]).strip().lower()
    if raw_answer not in {"да", "нет"}:
        raise ValueError(f"Unexpected RWSD answer: {ex['outputs']!r}")

    answer = "A" if raw_answer == "да" else "B"

    prompt = (
        f"Текст:\n{clean_text(inp.get('text'))}\n\n"
        f"Фрагмент 1: {clean_text(inp.get('span1_text'))}\n"
        f"Фрагмент 2: {clean_text(inp.get('span2_text'))}\n\n"
        "Относится ли фрагмент 2 к фрагменту 1?\n\n"
        "A) Да\n"
        "B) Нет"
    )

    return make_record(
        subset="rwsd",
        split=split,
        source_id=f"rwsd_{split}_{idx}",
        task_type="mcq",
        prompt=prompt,
        answer=answer,
        metadata={
            "id": meta.get("id", idx),
            "span1_index": inp.get("span1_index", None),
            "span2_index": inp.get("span2_index", None),
            "raw_answer": clean_text(ex["outputs"]),
            "option_a": "Да",
            "option_b": "Нет",
        },
    )



def format_use(ex: dict[str, Any], split: str, idx: int) -> dict[str, Any]:
    inp = ex["inputs"]
    meta = ex.get("meta", {})

    task = clean_text(inp.get("task"))
    text = clean_text(inp.get("text"))
    choices = clean_text(inp.get("choices"))
    additional_text = clean_text(inp.get("additional_text"))
    use_type = clean_text(meta.get("type"))

    parts = []
    if text:
        parts.append(f"Текст:\n{text}")
    if task:
        parts.append(f"Задание:\n{task}")
    if additional_text:
        parts.append(f"Дополнительный текст:\n{additional_text}")
    if choices:
        parts.append(f"Варианты ответа:\n{choices}")

    prompt = "\n\n".join(parts)
    if use_type.startswith("multiple_choice") or use_type == "matching":
        task_type = "numeric"
    else:
        task_type = "short_answer"

    return make_record(
        subset="use",
        split=split,
        source_id=f"use_{split}_{idx}",
        task_type=task_type,
        prompt=prompt,
        answer=clean_text(ex["outputs"]),
        metadata={
            "id": meta.get("id", idx),
            "id_task": clean_text(meta.get("id_task")),
            "variant": meta.get("variant", None),
            "score": meta.get("score", None),
            "type": use_type,
        },
    )

## Formatter registry

In [6]:
FORMATTERS = {
    "use": format_use,
    "mamuramu": format_mamuramu,
    "mathlogicqa": format_mathlogicqa,
    "ruopenbookqa": format_ruopenbookqa,
    "ruworldtree": format_ruworldtree,
    "parus": format_parus,
    "rcb": format_rcb,
    "rwsd": format_rwsd,
}

In [7]:
for task in SELECTED_TASKS:
    print(f"\n=== {task} ===")
    ds = load_dataset(HF_DATASET, task)
    print(ds)
    for split_name in ds.keys():
        print(f"  {split_name}: {len(ds[split_name])}")


=== use ===
DatasetDict({
    train: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 2622
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 900
    })
    validation: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 900
    })
})
  train: 2622
  test: 900
  validation: 900

=== mamuramu ===
DatasetDict({
    train: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 285
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 4248
    })
})
  train: 285
  test: 4248

=== mathlogicqa ===
DatasetDict({
    train: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 680
    })
    test: Dataset({
        features: ['instruction', 'inputs', 'outputs', 'meta'],
        num_rows: 1143
    })
})
  train: 680
  test: 114

# Normalized dataframe

In [8]:
# build normalized mera dataframe
rows = []
rejected_rows = []

for task in SELECTED_TASKS:
    print(f"\nProcessing {task}...")
    ds = load_dataset(HF_DATASET, task)
    formatter = FORMATTERS[task]

    for raw_split in ds.keys():
        if raw_split not in ALLOWED_SPLITS:
            print(f"  skipping split {raw_split}")
            continue

        out_split = "val" if raw_split in {"validation", "dev"} else "train"

        for idx, ex in enumerate(ds[raw_split]):
            try:
                record = formatter(ex, out_split, idx)

                if is_valid_record(record):
                    rows.append(record)
                else:
                    rejected_rows.append({
                        "subset": task,
                        "split": out_split,
                        "source_id": record.get("source_id", f"{task}_{out_split}_{idx}"),
                        "reason": "invalid_record",
                        "raw": to_json_str(ex),
                        "record": to_json_str(record),
                    })

            except Exception as e:
                rejected_rows.append({
                    "subset": task,
                    "split": out_split,
                    "source_id": f"{task}_{out_split}_{idx}",
                    "reason": repr(e),
                    "raw": to_json_str(ex),
                    "record": "",
                })

clean_raw = pd.DataFrame(rows)
rejected = pd.DataFrame(rejected_rows)

print("Raw clean rows:", len(clean_raw))
print("Rejected rows: ", len(rejected))

display(clean_raw.head())


Processing use...
  skipping split test

Processing mamuramu...
  skipping split test

Processing mathlogicqa...
  skipping split test

Processing ruopenbookqa...
  skipping split test

Processing ruworldtree...
  skipping split test

Processing parus...
  skipping split test

Processing rcb...
  skipping split test

Processing rwsd...
  skipping split test
Raw clean rows: 8908
Rejected rows:  0


,dataset,subset,language,split,source_id,task_type,prompt,answer,metadata
0,mera,use,russian,train,use_train_0,numeric,"Текст:\n(1) Один греческий историк по праву назвал Египет «даром Нила», который сделал Египет богатейшей житницей, кормившей население страны. (2) Люди здесь всегда селились на узких полосах земли по обоим берегам реки, несущей свои воды через сотни километров пустыни к дельте, где, разделившись на множество протоков, она впадает в Средиземное море. (3) Воды Нила ежегодно поднимались и опускались, оставляя в пойме слой плодородного ила, <...> позволяло строить сложные оросительные сооружения.\n\nЗадание:\nУкажите номера предложений, в которых верно передана ГЛАВНАЯ информация, содержащаяся в тексте. Запишите номера этих предложений.\n\nВарианты ответа:\n1) На берегах Нила всегда селились египтяне, потому что воды реки ежегодно поднимались и опускались, оставляя в пойме слой плодородного ила, в результате чего Египет стал богатейшей житницей и получил название “Дар Нила”\n2) Египтяне всегда селились на узких полосах земли по обоим берегам Нила, который нёс свои воды к дельте, где он впадал в Средиземное море\n3) Египет по праву назвали «даром Нила», так как на берегах этой реки селились египтяне и воды её, ежегодно поднимаясь и опускаясь, оставляли в пойме слой плодородного ила, что и сделало Египет богатейшей житницей\n4) Один греческий историк по праву назвал Египет «даром Нила», так как воды этой реки, ежегодно опускаясь, оставляли в пойме слой ила\n5) Египет стал колыбелью второй великой цивилизации в мировой истории, которая зародилась в долине Нила на узких полосах земли по обоим берегам реки","1,3","{'id': 0, 'id_task': '1', 'variant': 100, 'score': 1, 'type': 'multiple_choice_based_on_text'}"
1,mera,use,russian,train,use_train_1,numeric,"Текст:\n(1) Хохлома – старинный русский народный промысел, родившийся в XVII веке в Хохломской волости Семёновского уезда Нижегородской губернии и представляющий собой декоративную роспись деревянной посуды, шкатулок, мебели.(2) Роспись выглядит ярко, <…> тёмный фон: для создания рисунка используются такие цвета, как красный, жёлтый, оранжевый, зелёный, а также обязательно золотой.(3) После распада СССР хохломская роспись пережила трудные времена – сокращение числа мастеров (если в советский период промыслом было занято 2,5 тысячи человек, то к 2008 году 1,5 тысячи, а сейчас менее тысячи) , однако и в настоящее время работают фабрики, выпускающие изделия с хохломской росписью.\n\nЗадание:\nУкажите варианты ответов, в которых верно передана главная информация, содержащаяся в тексте. Запишите номера этих предложений.\n\nВарианты ответа:\n1. Хохлома – это русский народный промысел, возникший в ⅩⅦ веке в Нижегородской губернии, сохранившийся до нашего времени и представляющий собой декоративную роспись деревянных изделий, выполненную красными, жёлтыми, золотистыми и другими тонами по тёмному фону.\n2. Старинный русский народный промысел Хохлома представляет собой декоративную роспись деревянной посуды, шкатулок, мебели с помощью ярких красок на чёрном фоне; промысел появился в ⅩⅦ веке и существует до сих пор в Хохломской волости Семёновского уезда Нижегородской губернии, хотя число хохломских мастеров в наше время сократилось.\n3. Русский народный промысел, родившийся в ⅩⅦ веке в Хохломской волости Семёновского уезда Нижегородской губернии, представляет собой декоративную роспись деревянной посуды, шкатулок, мебели с помощью красного, жёлтого, золотого и некоторых других цветов; после распада СССР количество мастеров хохломской росписи уменьшилось, но производство сохраняется.\n4. Изделия с хохломской росписью, изготовляемые на Руси с ⅩⅦ века, выглядят ярко: для создания рисунка используются такие цвета, как красный, жёлтый, оранжевый, зелёный, золотой; традиционные элементы Хохломы – ягоды рябины и земляники, цветы и ветки, а также птицы, рыбы и звери.\n5. Хохломская роспись, родившаяся в ⅩⅦ веке и существующая в наше время, после распада СССР пережила трудные времена – с

In [9]:
# filter to mcq-like and numeric tasks only
KEEP_TASK_TYPES = {"mcq", "numeric", "binary_choice"}

clean = clean_raw[clean_raw["task_type"].isin(KEEP_TASK_TYPES)].copy()
excluded = clean_raw[~clean_raw["task_type"].isin(KEEP_TASK_TYPES)].copy()

print("Raw clean rows:", len(clean_raw))
print("Kept rows:     ", len(clean))
print("Excluded rows: ", len(excluded))

print("\nKept task types:")
display(
    clean.groupby(["subset", "task_type"])
    .size()
    .reset_index(name="n")
    .sort_values(["subset", "task_type"])
)

print("\nExcluded task types:")
if len(excluded):
    display(
        excluded.groupby(["subset", "task_type"])
        .size()
        .reset_index(name="n")
        .sort_values(["subset", "task_type"])
    )
else:
    print("No excluded rows.")

Raw clean rows: 8908
Kept rows:      7955
Excluded rows:  953

Kept task types:


,subset,task_type,n
0,mamuramu,mcq,285
1,mathlogicqa,mcq,680
2,parus,mcq,500
3,rcb,mcq,658
4,ruopenbookqa,mcq,2338
5,ruworldtree,mcq,115
6,rwsd,mcq,810
7,use,numeric,2569



Excluded task types:


,subset,task_type,n
0,use,short_answer,953


In [10]:
clean.head()

,dataset,subset,language,split,source_id,task_type,prompt,answer,metadata
0,mera,use,russian,train,use_train_0,numeric,"Текст:\n(1) Один греческий историк по праву назвал Египет «даром Нила», который сделал Египет богатейшей житницей, кормившей население страны. (2) Люди здесь всегда селились на узких полосах земли по обоим берегам реки, несущей свои воды через сотни километров пустыни к дельте, где, разделившись на множество протоков, она впадает в Средиземное море. (3) Воды Нила ежегодно поднимались и опускались, оставляя в пойме слой плодородного ила, <...> позволяло строить сложные оросительные сооружения.\n\nЗадание:\nУкажите номера предложений, в которых верно передана ГЛАВНАЯ информация, содержащаяся в тексте. Запишите номера этих предложений.\n\nВарианты ответа:\n1) На берегах Нила всегда селились египтяне, потому что воды реки ежегодно поднимались и опускались, оставляя в пойме слой плодородного ила, в результате чего Египет стал богатейшей житницей и получил название “Дар Нила”\n2) Египтяне всегда селились на узких полосах земли по обоим берегам Нила, который нёс свои воды к дельте, где он впадал в Средиземное море\n3) Египет по праву назвали «даром Нила», так как на берегах этой реки селились египтяне и воды её, ежегодно поднимаясь и опускаясь, оставляли в пойме слой плодородного ила, что и сделало Египет богатейшей житницей\n4) Один греческий историк по праву назвал Египет «даром Нила», так как воды этой реки, ежегодно опускаясь, оставляли в пойме слой ила\n5) Египет стал колыбелью второй великой цивилизации в мировой истории, которая зародилась в долине Нила на узких полосах земли по обоим берегам реки","1,3","{'id': 0, 'id_task': '1', 'variant': 100, 'score': 1, 'type': 'multiple_choice_based_on_text'}"
1,mera,use,russian,train,use_train_1,numeric,"Текст:\n(1) Хохлома – старинный русский народный промысел, родившийся в XVII веке в Хохломской волости Семёновского уезда Нижегородской губернии и представляющий собой декоративную роспись деревянной посуды, шкатулок, мебели.(2) Роспись выглядит ярко, <…> тёмный фон: для создания рисунка используются такие цвета, как красный, жёлтый, оранжевый, зелёный, а также обязательно золотой.(3) После распада СССР хохломская роспись пережила трудные времена – сокращение числа мастеров (если в советский период промыслом было занято 2,5 тысячи человек, то к 2008 году 1,5 тысячи, а сейчас менее тысячи) , однако и в настоящее время работают фабрики, выпускающие изделия с хохломской росписью.\n\nЗадание:\nУкажите варианты ответов, в которых верно передана главная информация, содержащаяся в тексте. Запишите номера этих предложений.\n\nВарианты ответа:\n1. Хохлома – это русский народный промысел, возникший в ⅩⅦ веке в Нижегородской губернии, сохранившийся до нашего времени и представляющий собой декоративную роспись деревянных изделий, выполненную красными, жёлтыми, золотистыми и другими тонами по тёмному фону.\n2. Старинный русский народный промысел Хохлома представляет собой декоративную роспись деревянной посуды, шкатулок, мебели с помощью ярких красок на чёрном фоне; промысел появился в ⅩⅦ веке и существует до сих пор в Хохломской волости Семёновского уезда Нижегородской губернии, хотя число хохломских мастеров в наше время сократилось.\n3. Русский народный промысел, родившийся в ⅩⅦ веке в Хохломской волости Семёновского уезда Нижегородской губернии, представляет собой декоративную роспись деревянной посуды, шкатулок, мебели с помощью красного, жёлтого, золотого и некоторых других цветов; после распада СССР количество мастеров хохломской росписи уменьшилось, но производство сохраняется.\n4. Изделия с хохломской росписью, изготовляемые на Руси с ⅩⅦ века, выглядят ярко: для создания рисунка используются такие цвета, как красный, жёлтый, оранжевый, зелёный, золотой; традиционные элементы Хохломы – ягоды рябины и земляники, цветы и ветки, а также птицы, рыбы и звери.\n5. Хохломская роспись, родившаяся в ⅩⅦ веке и существующая в наше время, после распада СССР пережила трудные времена – с

In [11]:
# inspect final counts
print("Counts by subset and split:")
display(
    clean.groupby(["subset", "split"])
    .size()
    .reset_index(name="n")
    .sort_values(["subset", "split"])
)

print("\nTotal by subset:")
display(
    clean.groupby("subset")
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

print("\nRejected rows:")
if len(rejected):
    display(
        rejected.groupby(["subset", "reason"])
        .size()
        .reset_index(name="n")
        .sort_values(["subset", "n"], ascending=[True, False])
    )
else:
    print("No rejected rows.")

Counts by subset and split:


,subset,split,n
0,mamuramu,train,285
1,mathlogicqa,train,680
2,parus,train,400
3,parus,val,100
4,rcb,train,438
5,rcb,val,220
6,ruopenbookqa,train,2338
7,ruworldtree,train,115
8,rwsd,train,606
9,rwsd,val,204



Total by subset:


,subset,n
7,use,2569
4,ruopenbookqa,2338
6,rwsd,810
1,mathlogicqa,680
3,rcb,658
2,parus,500
0,mamuramu,285
5,ruworldtree,115



Rejected rows:
No rejected rows.


In [12]:
# inspect a few final examples
def show_examples(df: pd.DataFrame, subset: str, n: int = 2):
    sub = df[df["subset"].str.lower() == subset.lower()]
    if sub.empty:
        print(f"No rows for {subset}")
        return

    sample = sub.sample(min(n, len(sub)), random_state=42)
    for _, row in sample.iterrows():
        print("=" * 80)
        print(f"subset: {row['subset']} | split: {row['split']} | task_type: {row['task_type']}")
        print("\nPROMPT:")
        print(row["prompt"])
        print("\nANSWER:")
        print(row["answer"])
        print("\nMETADATA:")
        print(row["metadata"])
        print()

for task in SELECTED_TASKS:
    if task in set(clean["subset"]):
        print(f"\n\n### {task}")
        show_examples(clean, task, n=2)



### use
subset: use | split: train | task_type: numeric

PROMPT:
Задание:
Укажите варианты ответов, в которых во всех словах одного ряда пропущена безударная чередующаяся гласная корня. Запишите номера ответов.

Варианты ответа:
1) хл..потать, зап..реть (на замок), р..сток
2) забл..стел, ск..калка, соб..рать
3) пож..мающий, соч..тание, поч..тать (старших)
4) взр..слеть, г..рнолыжный, оз..рение
5) попл..вок, зад..раться, изл..жить (суть проблемы)

ANSWER:
2,3,5

METADATA:
{'id': 1462, 'id_task': '9', 'variant': 61, 'score': 1, 'type': 'multiple_choice_independent_options'}

subset: use | split: train | task_type: numeric

PROMPT:
Текст:
(1) Многие считают понятие чести устарелым, несовременным, в том смысле, что оно нынче не применимо — не те условия. (2) Для одних это связано с такими действиями, как дуэль: мол, чем иначе можно защитить свою честь от оскорблений? (3) Другие считают: честь сегодня заменена более высоким понятием — принципиальность. (4) Вместо человека чести — человек 

In [13]:
# inspect binary-choice label distributions
binary = clean_raw[clean_raw["task_type"] == "binary_choice"].copy()

display(
    binary.groupby(["subset", "answer"])
    .size()
    .reset_index(name="n")
    .sort_values(["subset", "answer"])
)

display(
    binary.groupby(["subset", "split", "answer"])
    .size()
    .reset_index(name="n")
    .sort_values(["subset", "split", "answer"])
)

rwsd = clean_raw[clean_raw["subset"] == "rwsd"].copy()

display(
    rwsd.groupby(["split", "answer"])
    .size()
    .reset_index(name="n")
    .sort_values(["split", "answer"])
)

,subset,answer,n


,subset,split,answer,n


,split,answer,n
0,train,A,297
1,train,B,309
2,val,A,91
3,val,B,113


In [14]:
# check which hf splits were used
for task in SELECTED_TASKS:
    ds = load_dataset(HF_DATASET, task)
    allowed = [s for s in ds.keys() if s in ALLOWED_SPLITS]
    skipped = [s for s in ds.keys() if s not in ALLOWED_SPLITS]
    print(f"{task:15} allowed={allowed} skipped={skipped}")

use             allowed=['train', 'validation'] skipped=['test']
mamuramu        allowed=['train'] skipped=['test']
mathlogicqa     allowed=['train'] skipped=['test']
ruopenbookqa    allowed=['train'] skipped=['test']
ruworldtree     allowed=['train'] skipped=['test']
parus           allowed=['train', 'validation'] skipped=['test']
rcb             allowed=['train', 'validation'] skipped=['test']
rwsd            allowed=['train', 'validation'] skipped=['test']


In [15]:
for subset in ["parus", "rcb", "rwsd"]:
    sub = clean[clean["subset"] == subset]
    print(f"\n{subset}")
    display(sub["answer"].value_counts().sort_index())
    display(sub[["prompt", "answer", "metadata"]].sample(min(3, len(sub)), random_state=42))


parus


answer
A    250
B    250
Name: count, dtype: int64

,prompt,answer,metadata
7301,Мужчина зацепился рукавом за перекладину забора.\n\nЧто является следствием описанной ситуации?\n\nA) Он лез через забор.\nB) Рукав порвался.,B,"{'task': 'effect', 'id': 361, 'raw_answer': '2', 'option_a': 'Он лез через забор.', 'option_b': 'Рукав порвался.'}"
7013,Его одежда сидит слишком свободно.\n\nЧто является причиной описанной ситуации?\n\nA) Он купил её на распродаже.\nB) Он сбросил вес.,B,"{'task': 'cause', 'id': 73, 'raw_answer': '2', 'option_a': 'Он купил её на распродаже.', 'option_b': 'Он сбросил вес.'}"
7314,У человека наболело.\n\nЧто является причиной описанной ситуации?\n\nA) Его жена критикует его.\nB) Его мать скончалась.,B,"{'task': 'cause', 'id': 374, 'raw_answer': '2', 'option_a': 'Его жена критикует его.', 'option_b': 'Его мать скончалась.'}"



rcb


answer
A    219
B    219
C    220
Name: count, dtype: int64

,prompt,answer,metadata
8032,"Ситуация:\nЭто продолжится. - Каким вы видите дальнейшее развитие программы, кто - следующий? - Пока я этого сказать не могу.\n\nГипотеза:\nПрограмма будет развиваться в дальнейшем.\n\nВыберите отношение гипотезы к ситуации:\nA) гипотеза следует из ситуации\nB) гипотеза противоречит ситуации\nC) гипотеза независима от ситуации",C,"{'genre': 'interfax', 'verb': 'видеть', 'negation': 'no_negation', 'id': 154, 'raw_answer': '3', 'option_a': 'гипотеза следует из ситуации', 'option_b': 'гипотеза противоречит ситуации', 'option_c': 'гипотеза независима от ситуации'}"
7750,"Ситуация:\n— А я вот такой один на сто тысяч. Бабочка отвернулся, врать он не умел, но и не похвастаться тоже не мог, потому и увеличил цифру вдвое. К тому же он свято верил, что Господь обязательно накажет его, боялся этого, но все равно не смог удержаться ото лжи.\n\nГипотеза:\nГосподь обязательно накажет его.\n\nВыберите отношение гипотезы к ситуации:\nA) гипотеза следует из ситуации\nB) гипотеза противоречит ситуации\nC) гипотеза независима от ситуации",A,"{'genre': 'fiction', 'verb': 'верить,бояться', 'negation': 'no_negation', 'id': 310, 'raw_answer': '1', 'option_a': 'гипотеза следует из ситуации', 'option_b': 'гипотеза противоречит ситуации', 'option_c': 'гипотеза независима от ситуации'}"
7731,"Ситуация:\nВ итоге она, вместе с одной из коллег, отнесла гранату в кабинет директора школы. В итоге всех учеников вывели из школы и отпустили по домам. Позже в полиции определили, что принесенная в школу граната уже не представляет опасности, так как ее невозможно привести в действие.\n\nГипотеза:\nПринесенная в школу граната уже не представляет опасности, так как ее невозможно привести в действие.\n\nВыберите отношение гипотезы к ситуации:\nA) гипотеза следует из ситуации\nB) гипотеза противоречит ситуации\nC) гипотеза независима от ситуации",A,"{'genre': 'interfax', 'verb': 'определить', 'negation': 'negation', 'id': 291, 'raw_answer': '1', 'option_a': 'гипотеза следует из ситуации', 'option_b': 'гипотеза противоречит ситуации', 'option_c': 'гипотеза независима от ситуации'}"



rwsd


answer
A    388
B    422
Name: count, dtype: int64

,prompt,answer,metadata
8345,"Текст:\nСэм подвинул стул к роялю, но он был сломан, так что ему пришлось петь.\n\nФрагмент 1: роялю\nФрагмент 2: он\n\nОтносится ли фрагмент 2 к фрагменту 1?\n\nA) Да\nB) Нет",A,"{'id': 247, 'span1_index': 4, 'span2_index': 6, 'raw_answer': 'Да', 'option_a': 'Да', 'option_b': 'Нет'}"
8768,"Текст:\nПосле того, как я увидел, как Билли ловит мух и отрывает им крылья, я надрал ему уши. Я показал хозяину мух, некоторые из которых были раздавлены, а некоторые ползали в беспомощности, и я показал ему крылья на подоконнике. Я никогда раньше не видел его таким злым, но, поскольку Билли все еще выл и скулил, как трус, которым он и был, он больше не наказывал его, но посадил его на стул до конца дня, и сказал, что ему больше не разрешается играть на этой неделе.\n\nФрагмент 1: Билли\nФрагмент 2: его таким злым\n\nОтносится ли фрагмент 2 к фрагменту 1?\n\nA) Да\nB) Нет",B,"{'id': 64, 'span1_index': 6, 'span2_index': 43, 'raw_answer': 'Нет', 'option_a': 'Да', 'option_b': 'Нет'}"
8325,"Текст:\nСэм и Эми влюблены друг в друга, но родители Эми не рады этому. Это потому, что они беспокоятся.\n\nФрагмент 1: родители Эми\nФрагмент 2: они беспокоятся\n\nОтносится ли фрагмент 2 к фрагменту 1?\n\nA) Да\nB) Нет",A,"{'id': 227, 'span1_index': 8, 'span2_index': 16, 'raw_answer': 'Да', 'option_a': 'Да', 'option_b': 'Нет'}"


# Save

In [16]:
# save final processed files
clean_to_save = clean.copy()
excluded_to_save = excluded.copy()
rejected_to_save = rejected.copy()

clean_to_save["metadata"] = clean_to_save["metadata"].apply(to_json_str)

if len(excluded_to_save):
    excluded_to_save["metadata"] = excluded_to_save["metadata"].apply(to_json_str)

clean_path = PROCESSED_ROOT / "mera_clean_all.csv"
excluded_path = PROCESSED_ROOT / "mera_excluded_short_answer.csv"
rejected_path = PROCESSED_ROOT / "mera_rejected_all.csv"

clean_to_save.to_csv(clean_path, index=False, encoding="utf-8")
excluded_to_save.to_csv(excluded_path, index=False, encoding="utf-8")
rejected_to_save.to_csv(rejected_path, index=False, encoding="utf-8")

print(f"Saved clean:    {clean_path}")
print(f"Saved excluded: {excluded_path}")
print(f"Saved rejected: {rejected_path}")

for subset, sub in clean_to_save.groupby("subset"):
    out_path = PROCESSED_ROOT / f"mera_clean_{subset}.csv"
    sub.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved {subset}: {len(sub)} rows -> {out_path}")

Saved clean:    /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_all.csv
Saved excluded: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_excluded_short_answer.csv
Saved rejected: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_rejected_all.csv
Saved mamuramu: 285 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_mamuramu.csv
Saved mathlogicqa: 680 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_mathlogicqa.csv
Saved parus: 500 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_parus.csv
Saved rcb: 658 rows -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/mera/mera_clean_rcb.csv
Saved ruopenbookqa: 2338 rows -

# Final check

In [17]:
df = pd.read_csv(PROCESSED_ROOT / "mera_clean_all.csv")

display(df.groupby(["subset", "split"]).size().reset_index(name="n"))
display(df.groupby(["subset", "task_type"]).size().reset_index(name="n"))
display(df["task_type"].value_counts())


print(df.columns.tolist())
print("has source_id:", "source_id" in df.columns)
assert not (df["task_type"] == "short_answer").any()
assert set(df["language"]) == {"russian"}
assert df["prompt"].notna().all()
assert df["answer"].notna().all()

,subset,split,n
0,mamuramu,train,285
1,mathlogicqa,train,680
2,parus,train,400
3,parus,val,100
4,rcb,train,438
5,rcb,val,220
6,ruopenbookqa,train,2338
7,ruworldtree,train,115
8,rwsd,train,606
9,rwsd,val,204


,subset,task_type,n
0,mamuramu,mcq,285
1,mathlogicqa,mcq,680
2,parus,mcq,500
3,rcb,mcq,658
4,ruopenbookqa,mcq,2338
5,ruworldtree,mcq,115
6,rwsd,mcq,810
7,use,numeric,2569


task_type
mcq        5386
numeric    2569
Name: count, dtype: int64

['dataset', 'subset', 'language', 'split', 'source_id', 'task_type', 'prompt', 'answer', 'metadata']
has source_id: True
